# Amazon Bedrock — Cleaned Examples

This consolidated notebook collects the most useful Bedrock examples from the repository in a single, cleaned, and runnable document:

- Synchronous `InvokeModel` example
- Streaming `InvokeModelWithResponseStream` example
- Asynchronous `StartAsyncInvoke` example
- Batch `CreateModelInvocationJob` example

Notes:
- Keep AWS credentials out of the repo and configure them with the AWS CLI or environment variables.
- Set `AWS_REGION` and `BEDROCK_MODEL_ID` before running cells.

In [ ]:
# Cell 2: Imports and configuration helper
import os
import json
import time
import random
import boto3

region = os.getenv('AWS_REGION', 'us-east-1')
model_id = os.getenv('BEDROCK_MODEL_ID', 'amazon.nova-micro-v1:0')

bedrock = boto3.client('bedrock', region_name=region)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region)

print('Configured clients for region:', region)
print('Default model_id:', model_id)

## Synchronous InvokeModel

Sends a single request and waits for the complete response. Suitable for short, interactive prompts.

In [ ]:
# Cell 4: InvokeModel example
payload = {
    'inputText': 'Explain the difference between Bedrock control plane and data plane in two short bullets.',
    'textGenerationConfig': {
        'maxTokenCount': 200,
        'temperature': 0.4,
        'topP': 0.9,
    },
}

response = bedrock_runtime.invoke_model(
    modelId=model_id,
    body=json.dumps(payload),
)

try:
    result = json.loads(response['body'].read())
    print(json.dumps(result, indent=2))
except Exception:
    # Some models return slightly different shapes — print raw body for debugging
    print('Raw response body:')
    print(response['body'].read())

## Streaming InvokeModelWithResponseStream

Use streaming for improved interactivity and lower perceived latency in chat-like interfaces.

In [ ]:
# Cell 6: Streaming example
stream_payload = {
    'inputText': 'Write a three-item outline for a technical blog post on Bedrock endpoints.',
    'textGenerationConfig': {
        'maxTokenCount': 250,
        'temperature': 0.6,
        'topP': 0.95,
    },
}

stream_response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=model_id,
    body=json.dumps(stream_payload),
)

# Stream processing: many models stream JSON-chunks; assemble text conservatively
assembled = ''
for event in stream_response.get('body', []):
    if not isinstance(event, dict):
        continue
    chunk = event.get('chunk') or {}
    bytes_data = chunk.get('bytes')
    if not bytes_data:
        continue
    try:
        # Some SDKs deliver bytes; ensure string before parsing
        if isinstance(bytes_data, (bytes, bytearray)):
            chunk_text = bytes_data.decode('utf-8')
        else:
            chunk_text = bytes_data
        j = json.loads(chunk_text)
        # Common streaming delta keys vary by model; try multiple patterns
        text_piece = None
        if 'outputText' in j:
            text_piece = j.get('outputText')
        elif 'contentBlockDelta' in j:
            text_piece = j['contentBlockDelta'].get('delta', {}).get('text')
        elif 'chunk' in j and isinstance(j['chunk'], dict):
            text_piece = j['chunk'].get('outputText')
        if text_piece:
            assembled += text_piece
            print(text_piece, end='', flush=True)
    except Exception:
        # If parsing fails, print raw chunk as fallback
        print(chunk_text, end='', flush=True)
print('\n-- stream end --')

## Asynchronous StartAsyncInvoke

Submit long-running jobs (video, audio, large-generation) and poll or check status separately.

In [ ]:
# Cell 8: StartAsyncInvoke example (video)
async_payload = {
    'taskType': 'TEXT_VIDEO',
    'textToVideoParams': {'text': 'A calm sunrise over a mountain lake.'},
    'videoGenerationConfig': {
        'fps': 24,
        'durationSeconds': 6,
        'dimension': '1280x720',
        'seed': random.randint(0, 2147483646),
    },
}

output_config = {
    's3OutputDataConfig': {
        's3Uri': 's3://<bucket_name>/<prefix>/',
    },
}

async_response = bedrock_runtime.start_async_invoke(
    modelId='amazon.nova-reel-v1:0',
    modelInput=async_payload,
    outputDataConfig=output_config,
)
print('Async invocation ARN:', async_response.get('invocationArn'))

# Example: poll status (do not poll too frequently in production)
invocation_arn = async_response.get('invocationArn')
if invocation_arn:
    for i in range(10):
        status = bedrock_runtime.get_async_invoke(invocationArn=invocation_arn)
        print('Status:', status.get('status'))
        if status.get('status') in {'COMPLETED', 'FAILED', 'CANCELLED'}:
            break
        time.sleep(10)

## Batch CreateModelInvocationJob

Submit large datasets from S3 for batch processing and write results back to S3.

In [ ]:
# Cell 10: CreateModelInvocationJob example
# WARNING: ensure your roleArn and S3 buckets are correct and accessible.
job_response = bedrock.create_model_invocation_job(
    roleArn='arn:aws:iam::<account>:role/<bedrock-invoke-role>',
    modelId='anthropic.claude-3-haiku-20240307-v1:0',
    jobName='bedrock-batch-demo',
    inputDataConfig={
        's3InputDataConfig': {'s3Uri': 's3://<bucket_name>/<input_file>.jsonl'},
    },
    outputDataConfig={
        's3OutputDataConfig': {'s3Uri': 's3://<bucket_name>/<output-prefix>/'},
    },
)
print('Job ARN:', job_response.get('jobArn'))

## Best Practices and Notes

- Use explicit model IDs and document which model/version you used.  
- Implement exponential backoff and retry logic for throttling.  
- Sanitize input data and redact sensitive information before sending to Bedrock.  
- Track ARNs and outputs for auditability.  
- Monitor costs when running heavy or long jobs.